# 08. Теория информации: энтропия, кросс-энтропия, KL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/08_information_theory.ipynb)

Код из раздела [modules/08_information_theory.md](https://github.com/justxor/math-for-ai-2026/blob/main/modules/08_information_theory.md#m08). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### 🏋️ Практика модуля

**8.3.** Посчитайте KL в обе стороны для P = [0.9, 0.1], Q = [0.5, 0.5].

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import numpy as np
P, Q = np.array([0.9, 0.1]), np.array([0.5, 0.5])
kl = lambda a, b: np.sum(a * np.log(a / b))
print(kl(P, Q), kl(Q, P))   # 0.368, 0.511 — разные

## 🔬 Лаборатория: иллюстрации модуля

Это код картинок из курса. Меняйте параметры (learning rate, температуру, число точек, ядро…) и перезапускайте ячейку — так интуиция появляется быстрее всего.

In [ ]:
# Помощники для иллюстраций (те же, что в scripts/make_figures.py)
import matplotlib.pyplot as plt
from scipy import stats
C = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def save(fig, name):          # в ноутбуке просто показываем картинку
    fig.tight_layout(); plt.show()

def arrow(ax, v, color, label=None, origin=(0, 0), **kw):
    ax.annotate("", xy=(origin[0] + v[0], origin[1] + v[1]), xytext=origin,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, **kw))
    if label:
        ax.text(origin[0] + v[0] * 1.08, origin[1] + v[1] * 1.08, label, color=color, fontsize=13, fontweight="bold")

def rosen_like(x, y):
    return 0.5 * x**2 + 4 * y**2

In [ ]:
def entropy_kl():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    p = np.linspace(1e-4, 1 - 1e-4, 400)
    H = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))
    axes[0].plot(p, H, color=C[0], lw=2.5)
    axes[0].set_xlabel("p (вероятность «орла»)"); axes[0].set_ylabel("H, бит")
    axes[0].set_title("Энтропия монеты: максимум 1 бит при p = 0.5")
    x = np.linspace(-6, 8, 500)
    P = 0.6 * stats.norm.pdf(x, -1.5, 0.8) + 0.4 * stats.norm.pdf(x, 3, 1)
    Q = stats.norm.pdf(x, 0.3, 2.4)
    axes[1].fill_between(x, P, alpha=0.3, color=C[0], label="P (данные)")
    axes[1].plot(x, Q, color=C[1], lw=2, label="Q (модель)")
    dx = x[1] - x[0]
    kl_pq = np.sum(P * np.log((P + 1e-12) / (Q + 1e-12))) * dx
    kl_qp = np.sum(Q * np.log((Q + 1e-12) / (P + 1e-12))) * dx
    axes[1].set_title(f"KL несимметрична: KL(P‖Q) = {kl_pq:.2f},  KL(Q‖P) = {kl_qp:.2f}")
    axes[1].legend()
    save(fig, "entropy_kl")

entropy_kl()

In [ ]:
def kl_fit():
    x = np.linspace(-8, 8, 2000); dx = x[1] - x[0]
    P = 0.5 * stats.norm.pdf(x, -2.5, 0.8) + 0.5 * stats.norm.pdf(x, 2.5, 0.8)
    best_f, best_r = None, None
    for mu in np.linspace(-4, 4, 81):
        for s in np.linspace(0.3, 4, 75):
            Q = stats.norm.pdf(x, mu, s) + 1e-300
            f = np.sum(P * np.log((P + 1e-300) / Q)) * dx
            r = np.sum(Q * np.log(Q / (P + 1e-300))) * dx
            if best_f is None or f < best_f[0]: best_f = (f, mu, s)
            if best_r is None or r < best_r[0]: best_r = (r, mu, s)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
    for ax, (val, mu, s), t, c in [(axes[0], best_f, "min KL(P‖Q) — forward: накрывает обе моды", C[0]),
                                   (axes[1], best_r, "min KL(Q‖P) — reverse: садится на одну моду", C[1])]:
        ax.fill_between(x, P, color="gray", alpha=0.3, label="P (данные)")
        ax.plot(x, stats.norm.pdf(x, mu, s), color=c, lw=2.5, label=f"Q = N({mu:.1f}, {s:.2f}²)")
        ax.set_title(t, fontsize=11); ax.legend(fontsize=9)
    save(fig, "kl_forward_reverse")

kl_fit()